<a href="https://colab.research.google.com/github/eikarna/notebooks/blob/main/stable_cascade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# Update repo & packages
!apt update && apt upgrade -y

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
All packages are up to date.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Calculating upgrade... Done
0 upgraded, 0 newly insta

In [15]:
# Langkah 1: Install dependensi yang diperlukan
# (Jalankan perintah berikut di Google Colab atau Kaggle jika belum terinstall)
!pip install transformers accelerate gradio diffusers
!pip install torch torch_xla[tpu]~=2.6.0 -f https://storage.googleapis.com/libtpu-releases/index.html

Looking in links: https://storage.googleapis.com/libtpu-releases/index.html


In [ ]:
# Cek PyTorch support TPU atau ngga
!PJRT_DEVICE=TPU python3 -c "import torch_xla.core.xla_model as xm; print(xm.get_xla_supported_devices(\"TPU\"))"

In [ ]:
# Refresh TPU
!pkill -f python

In [12]:

import json
import os
import random
import shutil
import warnings
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple
from uuid import uuid4

import gradio as gr
import numpy as np
import torch
import torch_xla
import torch_xla.core.xla_model as xm
import PIL.Image
from diffusers import StableCascadeDecoderPipeline, StableCascadePriorPipeline
from diffusers.pipelines.wuerstchen import DEFAULT_STAGE_C_TIMESTEPS
from filelock import FileLock
from PIL.Image import Image

# -----------------------------------------------------------------------------
# TPU Setup
# -----------------------------------------------------------------------------
try:
    import torch_xla
    DEVICE = xm.xla_device()
    TPU_AVAILABLE = True
except ImportError:
    TPU_AVAILABLE = False
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------------------------------------------------------
# Bagian User History yang dimodifikasi untuk Colab
# -----------------------------------------------------------------------------

class ColabHistory:
    def __init__(self):
        self.folder_path = Path("./colab_history")
        self.images_path = self.folder_path / "images"
        self.metadata_file = self.folder_path / "history.jsonl"
        self.lock = FileLock(self.folder_path / "history.lock")

        self.folder_path.mkdir(parents=True, exist_ok=True)
        self.images_path.mkdir(parents=True, exist_ok=True)

    def save_image(self, image: Image, prompt: str, metadata: Dict):
        """Menyimpan gambar dan metadata ke folder history"""
        image_path = self._save_image_file(image)
        data = {
            "path": str(image_path),
            "prompt": prompt,
            "metadata": {
                "datetime": str(datetime.now()),
                **metadata
            }
        }

        with self.lock:
            with open(self.metadata_file, "a") as f:
                f.write(json.dumps(data) + "\n")

    def load_history(self) -> List[Tuple[str, str]]:
        """Memuat seluruh history generate"""
        if not self.metadata_file.exists():
            return []

        history = []
        with self.lock:
            with open(self.metadata_file, "r") as f:
                for line in f.readlines():
                    data = json.loads(line)
                    history.append((data["path"], data["prompt"]))
        return list(reversed(history))

    def delete_history(self):
        """Menghapus semua history"""
        with self.lock:
            if self.images_path.exists():
                shutil.rmtree(self.images_path)
            if self.metadata_file.exists():
                os.remove(self.metadata_file)
            self.images_path.mkdir()

    def _save_image_file(self, image: Image) -> Path:
        """Menyimpan file gambar ke folder images"""
        filename = f"{uuid4().hex}.png"
        dst = self.images_path / filename
        image.save(dst)
        return dst

# Inisialisasi sistem history
colab_history = ColabHistory()

# -----------------------------------------------------------------------------
# Bagian Main Model
# -----------------------------------------------------------------------------
DESCRIPTION = "# Stable Cascade - Colab Version (TPU/GPU Support)"
DESCRIPTION += "\n<p style='text-align: center'>Unofficial demo for <a href='https://huggingface.co/stabilityai/stable-cascade' target='_blank'>Stable Cascade</a></p>"
MAX_SEED = np.iinfo(np.int32).max
MAX_IMAGE_SIZE = 2048
dtype = torch.bfloat16  # bfloat16 lebih optimal untuk TPU

# Inisialisasi model
try:
    # Prior Pipeline
    prior_pipeline = StableCascadePriorPipeline.from_pretrained(
        "stabilityai/stable-cascade-prior",
        torch_dtype=dtype
    )

    # Decoder Pipeline
    decoder_pipeline = StableCascadeDecoderPipeline.from_pretrained(
        "stabilityai/stable-cascade",
        torch_dtype=dtype
    )

    # Pindahkan model ke device
    prior_pipeline = prior_pipeline.to(DEVICE)
    decoder_pipeline = decoder_pipeline.to(DEVICE)

    # Optimasi untuk TPU
    if TPU_AVAILABLE:
        import torch_xla.debug.metrics as met
        torch.set_default_tensor_type('torch.FloatTensor')
        prior_pipeline = torch_xla.utils.checkpoint.checkpoint(prior_pipeline)
        decoder_pipeline = torch_xla.utils.checkpoint.checkpoint(decoder_pipeline)

except Exception as e:
    print(f"Error initializing model: {str(e)}")
    prior_pipeline = None
    decoder_pipeline = None

# -----------------------------------------------------------------------------
# Fungsi Generate
# -----------------------------------------------------------------------------

def randomize_seed_fn(seed: int, randomize_seed: bool) -> int:
    if randomize_seed:
        return random.randint(0, MAX_SEED)
    return seed


def generate(
    prompt: str,
    negative_prompt: str = "",
    seed: int = 0,
    width: int = 1024,
    height: int = 1024,
    prior_num_inference_steps: int = 30,
    prior_guidance_scale: float = 4.0,
    decoder_num_inference_steps: int = 12,
    decoder_guidance_scale: float = 0.0,
):
    if prior_pipeline is None or decoder_pipeline is None:
        raise RuntimeError("Model tidak terinisialisasi dengan benar!")

    # Generator untuk TPU/GPU
    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    # Prior stage dengan optimasi memory TPU
    with torch.no_grad():
        prior_output = prior_pipeline(
            prompt=prompt,
            height=height,
            width=width,
            num_inference_steps=prior_num_inference_steps,
            negative_prompt=negative_prompt,
            guidance_scale=prior_guidance_scale,
            generator=generator,
        )

    # Decoder stage dengan optimasi memory
    with torch.no_grad():
        decoder_output = decoder_pipeline(
            image_embeddings=prior_output.image_embeddings,
            prompt=prompt,
            num_inference_steps=decoder_num_inference_steps,
            guidance_scale=decoder_guidance_scale,
            negative_prompt=negative_prompt,
            generator=generator,
            output_type="pil",
        ).images[0]

    # Simpan ke history
    metadata = {
        "seed": seed,
        "width": width,
        "height": height,
        "negative_prompt": negative_prompt,
        "prior_guidance_scale": prior_guidance_scale,
        "decoder_guidance_scale": decoder_guidance_scale,
        "steps": {
            "prior": prior_num_inference_steps,
            "decoder": decoder_num_inference_steps
        }
    }

    colab_history.save_image(decoder_output, prompt, metadata)

    return decoder_output

# -----------------------------------------------------------------------------
# Antarmuka Gradio
# -----------------------------------------------------------------------------

examples = [
    "An astronaut riding a green horse",
    "A mecha robot in a favela by Tarsila do Amaral",
    "The spirit of a Tamagotchi wandering in the city of Los Angeles",
    "A delicious feijoada ramen dish",
]

css = '''
h1 { text-align: center; }
.gradio-container { max-width: 760px !important; }
#gallery { min-height: 600px; }
'''

with gr.Blocks(css=css) as demo:
    gr.Markdown(DESCRIPTION)

    with gr.Tabs():
        with gr.Tab("Generate"):
            with gr.Row():
                with gr.Column():
                    prompt_input = gr.Textbox(
                        label="Prompt",
                        placeholder="Enter your prompt...",
                        lines=3
                    )
                    negative_prompt_input = gr.Textbox(
                        label="Negative Prompt",
                        placeholder="What to exclude...",
                        lines=2
                    )

                    with gr.Row():
                        seed_input = gr.Slider(0, MAX_SEED, label="Seed", value=0)
                        random_seed = gr.Checkbox(label="Random Seed", value=True)

                    with gr.Row():
                        width_input = gr.Slider(0, MAX_IMAGE_SIZE, 1024, step=512, label="Width")
                        height_input = gr.Slider(0, MAX_IMAGE_SIZE, 1024, step=512, label="Height")

                    with gr.Row():
                        prior_guidance = gr.Slider(0, 20, 4.0, step=0.1, label="Prior Guidance")
                        decoder_guidance = gr.Slider(0, 20, 0.0, step=0.1, label="Decoder Guidance")

                    with gr.Row():
                        prior_steps = gr.Slider(0, 30, 20, step=1, label="Prior Steps")
                        decoder_steps = gr.Slider(0, 12, 10, step=1, label="Decoder Steps")

                    generate_btn = gr.Button("Generate", variant="primary")

                with gr.Column():
                    output_image = gr.Image(label="Result", interactive=False)

        with gr.Tab("History"):
            with gr.Row():
                refresh_btn = gr.Button("Refresh")
                clear_btn = gr.Button("Clear History", variant="stop")
            gallery = gr.Gallery(label="Generated Images")

    # Contoh-contoh
    gr.Examples(
        examples=examples,
        inputs=prompt_input,
        outputs=output_image,
        fn=generate,
        cache_examples=False
    )

    # Event handlers
    inputs = [
        prompt_input,
        negative_prompt_input,
        seed_input,
        width_input,
        height_input,
        prior_steps,
        prior_guidance,
        decoder_steps,
        decoder_guidance
    ]

    generate_btn.click(
        fn=randomize_seed_fn,
        inputs=[seed_input, random_seed],
        outputs=seed_input
    ).then(
        fn=generate,
        inputs=inputs,
        outputs=output_image
    )

    refresh_btn.click(
        fn=lambda: colab_history.load_history(),
        outputs=gallery
    )

    clear_btn.click(
        fn=lambda: colab_history.delete_history(),
        outputs=gallery
    )

    # Muat history saat pertama kali dibuka
    demo.load(
        fn=lambda: colab_history.load_history(),
        outputs=gallery
    )

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.39G [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Error initializing model: Bad StatusOr access: RESOURCE_EXHAUSTED: Error allocating device buffer: Attempting to allocate 12.50M. That was not possible. There are 10.81M free.; (0x0x0_HBM0)
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8861c29855ed5504e5.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/blocks.py", line 2137, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/blocks.py", line 1663, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/anyio/to_thread.py", line 56, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7862 <> https://8861c29855ed5504e5.gradio.live
